# Proyecto Avanzado Python

## Limpieza de Informacion

### 06_Metas

In [1]:
# Importaciones y configuración
import pandas as pd
import numpy as np
import os

ARCHIVO_ENTRADA = "/content/sample_data/06_metas.xlsx"          # ajusta la ruta según donde tengas el archivo
ARCHIVO_SALIDA = "/content/sample_data/06_metas_limpio.xlsx"

N_SUCURSALES = 12
MESES_POR_SUCURSAL = 16  # enero 2024 -> abril 2025



In [2]:
# Carga de datos
def cargar_datos(path):
    return pd.read_excel(path)

In [3]:
# Eliminacion de Duplicados
def eliminar_duplicados(df):
    antes = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"[1] Duplicados eliminados: {antes - len(df)} (quedan {len(df)} filas)")
    return df

In [4]:
# Reconstruir SucursalID y Fecha por posición
def reconstruir_sucursal_y_fecha(df):
    """
    La tabla está ordenada en bloques de 16 filas por sucursal (SUC-001 -> SUC-012),
    y dentro de cada bloque los meses van de 2024-01 a 2025-04 en orden.
    Por eso se reconstruyen ambas columnas 100% por posición.
    """
    esperado = N_SUCURSALES * MESES_POR_SUCURSAL
    if len(df) != esperado:
        raise ValueError(
            f"Se esperaban {esperado} filas tras eliminar duplicados, "
            f"pero hay {len(df)}. Revisar antes de reconstruir por posición."
        )

    fechas_bloque = pd.date_range("2024-01-01", periods=MESES_POR_SUCURSAL, freq="MS")
    sucursales, fechas = [], []
    for i in range(N_SUCURSALES):
        sucursales.extend([f"SUC-{i+1:03d}"] * MESES_POR_SUCURSAL)
        fechas.extend(fechas_bloque)

    df["SucursalID"] = sucursales
    df["Fecha"] = fechas
    print(f"[2] SucursalID y Fecha reconstruidos por posición para {len(df)} filas")
    return df

In [5]:
# Convertir columnas a numérico
def convertir_a_numerico(df):
    for col in ["Meta_Ventas", "Meta_Unidades"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")  # texto invalido -> NaN
    print("[3] Meta_Ventas y Meta_Unidades convertidas a numérico")
    return df

In [6]:
# Detectar inválidos con IQR (por sucursal) y calcular promedios
def calcular_limites_iqr(serie_positiva):
    """IQR calculado solo sobre valores positivos (ya excluye NaN, negativos y cero)."""
    q1, q3 = serie_positiva.quantile(0.25), serie_positiva.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def marcar_invalidos_y_calcular_promedios(df, columna):
    """
    Devuelve:
      - máscara booleana de inválidos (NaN, <=0 o fuera de rango IQR) por sucursal
      - dict {SucursalID: promedio_válido} para imputar
    """
    invalidos_total = pd.Series(False, index=df.index)
    promedios = {}

    for suc, grupo in df.groupby("SucursalID"):
        idx = grupo.index
        col = df.loc[idx, columna]

        invalidos = col.isna() | (col <= 0)

        positivos = col[~invalidos]
        if len(positivos) >= 4:  # con pocos datos el IQR no es confiable
            low, high = calcular_limites_iqr(positivos)
            fuera_rango = (~invalidos) & ((col < low) | (col > high))
            invalidos = invalidos | fuera_rango

        invalidos_total.loc[idx] = invalidos
        promedio_valido = col[idx][~invalidos].mean()
        promedios[suc] = promedio_valido

    return invalidos_total, promedios

In [7]:
# Imputar los valores inválidos
def imputar(df):
    reporte = {}
    for columna in ["Meta_Ventas", "Meta_Unidades"]:
        invalidos, promedios = marcar_invalidos_y_calcular_promedios(df, columna)
        reporte[columna] = int(invalidos.sum())

        for suc, promedio in promedios.items():
            mask = invalidos & (df["SucursalID"] == suc)
            df.loc[mask, columna] = round(promedio)

    print(f"[4-5] Valores inválidos imputados con promedio de su sucursal: {reporte}")
    return df

In [8]:
# Validar el resultado final
def validar_resultado(df):
    assert df.isna().sum().sum() == 0, "Quedaron valores nulos sin imputar"
    assert (df["Meta_Ventas"] <= 0).sum() == 0, "Quedaron Meta_Ventas <= 0"
    assert (df["Meta_Unidades"] <= 0).sum() == 0, "Quedaron Meta_Unidades <= 0"
    assert len(df) == N_SUCURSALES * MESES_POR_SUCURSAL, "Cambió la cantidad de filas"
    print("[6] Validación final OK: sin nulos, sin negativos/cero, sin filas perdidas")

In [9]:
# Función principal
def main():
    df = cargar_datos(ARCHIVO_ENTRADA)
    df = eliminar_duplicados(df)
    df = reconstruir_sucursal_y_fecha(df)
    df = convertir_a_numerico(df)
    df = imputar(df)

    df["Meta_Ventas"] = df["Meta_Ventas"].astype(int)
    df["Meta_Unidades"] = df["Meta_Unidades"].astype(int)

    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.head(20).to_string())

[1] Duplicados eliminados: 2 (quedan 192 filas)
[2] SucursalID y Fecha reconstruidos por posición para 192 filas
[3] Meta_Ventas y Meta_Unidades convertidas a numérico
[4-5] Valores inválidos imputados con promedio de su sucursal: {'Meta_Ventas': 19, 'Meta_Unidades': 15}
[6] Validación final OK: sin nulos, sin negativos/cero, sin filas perdidas

Archivo limpio guardado en: /content/sample_data/06_metas_limpio.xlsx
   SucursalID      Fecha  Meta_Ventas  Meta_Unidades
0     SUC-001 2024-01-01      6774353             38
1     SUC-001 2024-02-01      3579350             14
2     SUC-001 2024-03-01      5412798             23
3     SUC-001 2024-04-01      2460269              7
4     SUC-001 2024-05-01      3550319             10
5     SUC-001 2024-06-01      5612664             29
6     SUC-001 2024-07-01      3636956             13
7     SUC-001 2024-08-01       777559              6
8     SUC-001 2024-09-01      3916423             17
9     SUC-001 2024-10-01      4276721             14

### 05_Sucursales

In [10]:
# Configuración y carga
import pandas as pd

ARCHIVO_ENTRADA = "/content/sample_data/05_sucursales.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/05_sucursales_limpio.xlsx"

ZONAS_VALIDAS = {"Norte", "Sur", "Centro", "Digital"}


def cargar_datos(path):
    return pd.read_excel(path)

In [11]:
# Detectar valores inválidos en Zona
def detectar_zona_invalida(df):
    """
    Invalido = vacio (NaN) o cualquier valor que no pertenezca
    al set de categorias validas conocidas.
    """
    return df["Zona"].isna() | (~df["Zona"].isin(ZONAS_VALIDAS))

In [12]:
# Reemplazar inválidos: por ciudad → por país → moda global
def reemplazar_zona_invalida(df):
    invalidos = detectar_zona_invalida(df)
    print(f"[1] Zonas inválidas detectadas: {invalidos.sum()}")

    moda_global = df.loc[~invalidos, "Zona"].mode().iloc[0]

    for idx in df[invalidos].index:
        ciudad = df.loc[idx, "Ciudad"]
        pais = df.loc[idx, "Pais"]

        # Paso 1: buscar otra sucursal en la misma ciudad con zona valida
        candidatos_ciudad = df[(df["Ciudad"] == ciudad) & (~invalidos)]["Zona"]

        if len(candidatos_ciudad) > 0:
            nueva_zona = candidatos_ciudad.mode().iloc[0]
            criterio = f"misma ciudad ({ciudad})"
        else:
            # Paso 2: moda de Zona en el mismo pais
            candidatos_pais = df[(df["Pais"] == pais) & (~invalidos)]["Zona"]
            if len(candidatos_pais) > 0:
                nueva_zona = candidatos_pais.mode().iloc[0]
                criterio = f"moda del país ({pais})"
            else:
                # Paso 3: moda global como ultimo recurso
                nueva_zona = moda_global
                criterio = "moda global"

        df.loc[idx, "Zona"] = nueva_zona
        print(f"    -> {df.loc[idx, 'SucursalID']}: Zona reemplazada por '{nueva_zona}' (criterio: {criterio})")

    return df

In [13]:
# Validación final
def validar_resultado(df):
    assert df["Zona"].isin(ZONAS_VALIDAS).all(), "Aún quedan zonas inválidas"
    assert len(df) == 12, "Cambió la cantidad de filas (no debía eliminarse ninguna)"
    print("[2] Validación final OK: todas las zonas son válidas, no se eliminó ninguna fila")

In [14]:
# Función principal
def main():
    df = cargar_datos(ARCHIVO_ENTRADA)
    df = reemplazar_zona_invalida(df)
    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.to_string())

[1] Zonas inválidas detectadas: 1
    -> SUC-011: Zona reemplazada por 'Sur' (criterio: misma ciudad (Monterrey))
[2] Validación final OK: todas las zonas son válidas, no se eliminó ninguna fila

Archivo limpio guardado en: /content/sample_data/05_sucursales_limpio.xlsx
   SucursalID                 Sucursal       Pais        Ciudad     Zona    Estado
0     SUC-001      Sucursal Santiago 1      Chile      Santiago      Sur    Activa
1     SUC-002          Sucursal Lima 2       Perú          Lima    Norte    Activa
2     SUC-003       Sucursal Córdoba 3  Argentina       Córdoba    Norte    Activa
3     SUC-004    Sucursal Valparaíso 4      Chile    Valparaíso   Centro    Activa
4     SUC-005  Sucursal Buenos Aires 5  Argentina  Buenos Aires  Digital    Activa
5     SUC-006     Sucursal Monterrey 6     México     Monterrey      Sur    Activa
6     SUC-007      Sucursal Trujillo 7       Perú      Trujillo  Digital    Activa
7     SUC-008        Sucursal Bogotá 8   Colombia        Bogotá  

### 04_Vendedores

In [15]:
# Configuración y carga
import pandas as pd

ARCHIVO_ENTRADA = "/content/sample_data/04_vendedores.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/04_vendedores_limpio.xlsx"


def cargar_datos(path):
    return pd.read_excel(path)

In [16]:
# Estandarizar el formato de los nombres
def estandarizar_nombres(df):
    """
    Normaliza a formato 'Nombre Apellido' (primera letra en mayuscula),
    corrigiendo casos como 'maría torres' o 'javier torres'.
    """
    antes = df["Vendedor"].copy()
    df["Vendedor"] = df["Vendedor"].str.strip().str.title()

    cambios = (antes != df["Vendedor"]).sum()
    print(f"[1] Nombres estandarizados: {cambios} corregidos")
    return df

In [17]:
# Reemplazar valores faltantes en Nivel
def reemplazar_nivel_faltante(df):
    faltantes = df["Nivel"].isna().sum()
    df["Nivel"] = df["Nivel"].fillna("Sin nivel")
    print(f"[2] Valores faltantes en Nivel reemplazados por 'Sin nivel': {faltantes}")
    return df

In [18]:
# Validación final
def validar_resultado(df):
    assert df["Nivel"].isna().sum() == 0, "Aún quedan valores nulos en Nivel"
    assert len(df) == 30, "Se perdieron o agregaron vendedores (deben mantenerse todos)"
    print("[3] Validación final OK: sin nulos en Nivel, se mantienen los 30 vendedores")

In [19]:
# Función principal
def main():
    df = cargar_datos(ARCHIVO_ENTRADA)
    df = estandarizar_nombres(df)
    df = reemplazar_nivel_faltante(df)
    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.to_string())

[1] Nombres estandarizados: 2 corregidos
[2] Valores faltantes en Nivel reemplazados por 'Sin nivel': 1
[3] Validación final OK: sin nulos en Nivel, se mantienen los 30 vendedores

Archivo limpio guardado en: /content/sample_data/04_vendedores_limpio.xlsx
   VendedorID           Vendedor SucursalID        Nivel    Estado
0     VEN-001         Luis Rojas    SUC-012       Senior    Activo
1     VEN-002       Luis Morales    SUC-005       Junior    Activo
2     VEN-003        Sofía Silva    SUC-007       Senior    Activo
3     VEN-004        Pedro Gómez    SUC-003    Sin nivel    Activo
4     VEN-005        Luis Torres    SUC-012       Senior    Activo
5     VEN-006       Carlos Pérez    SUC-009  Semi Senior  Licencia
6     VEN-007    Valentina Pérez    SUC-005       Senior    Activo
7     VEN-008     Carlos Morales    SUC-001       Junior    Activo
8     VEN-009        Sofía Rojas    SUC-007       Junior  Retirado
9     VEN-010        Pedro Pérez    SUC-012       Senior  Licencia
10    V

### 03_Productos

In [20]:
# Configuración
import pandas as pd
import numpy as np
import re

ARCHIVO_ENTRADA = "/content/sample_data/03_productos.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/03_productos_limpio.xlsx"

CATEGORIAS_VALIDAS = {"Moda", "Hogar", "Oficina", "Tecnología", "Deportes", "Belleza"}
PATRON_PRODUCTO = re.compile(r"^\s*([A-Za-zÀ-ÿ]+)\s+Producto\s+(\d+)\s*$", re.IGNORECASE)
PATRON_PRODUCTOID = re.compile(r"^PROD-(\d+)$", re.IGNORECASE)


def cargar_datos(path):
    return pd.read_excel(path)

In [21]:
# Eliminar duplicados
def eliminar_duplicados(df):
    antes = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"[1] Duplicados eliminados: {antes - len(df)} (quedan {len(df)} filas)")
    return df

In [22]:
# Funciones auxiliares para extraer categoría/número desde texto
def _extraer_de_producto(valor):
    """Devuelve (categoria, numero) si el nombre del producto matchea el patron 'Categoria Producto N'."""
    if not isinstance(valor, str):
        return None, None
    m = PATRON_PRODUCTO.match(valor.strip())
    if not m:
        return None, None
    categoria_cruda, numero = m.group(1), int(m.group(2))
    for cat_valida in CATEGORIAS_VALIDAS:
        if categoria_cruda.lower() == cat_valida.lower():
            return cat_valida, numero
    return None, None


def _extraer_numero_de_id(valor):
    if not isinstance(valor, str):
        return None
    m = PATRON_PRODUCTOID.match(valor.strip())
    return int(m.group(1)) if m else None

In [23]:
# Reconstrucción cruzada de Producto, Categoria y ProductoID
def reconstruir_producto_categoria_id(df):
    categorias_finales, productos_finales, numeros_finales = [], [], []
    reconstruidos_producto = 0
    reconstruidos_categoria = 0

    for _, fila in df.iterrows():
        cat_desde_producto, num_desde_producto = _extraer_de_producto(fila["Producto"])

        if cat_desde_producto is not None:
            # El nombre del producto es valido: manda sobre la Categoria guardada
            categoria = cat_desde_producto
            numero = num_desde_producto
            producto = fila["Producto"].strip()
            if fila["Categoria"] != categoria:
                reconstruidos_categoria += 1
        else:
            # Falta o esta corrupto el nombre: se reconstruye desde Categoria + numero del ProductoID
            categoria_original = fila["Categoria"]
            categoria = categoria_original if categoria_original in CATEGORIAS_VALIDAS else None
            numero = _extraer_numero_de_id(fila["ProductoID"])

            if categoria is not None and numero is not None:
                producto = f"{categoria} Producto {numero}"
                reconstruidos_producto += 1
            else:
                categoria = categoria_original
                producto = fila["Producto"]

        categorias_finales.append(categoria)
        productos_finales.append(producto)
        numeros_finales.append(numero)

    df["Categoria"] = categorias_finales
    df["Producto"] = productos_finales
    numeros = pd.Series(numeros_finales, index=df.index)
    df["ProductoID"] = numeros.apply(lambda n: f"PROD-{int(n):04d}" if pd.notna(n) else np.nan)

    print(f"[2] Categoria reconstruida/corregida en {reconstruidos_categoria} filas")
    print(f"[3] Producto reconstruido en {reconstruidos_producto} filas")
    print(f"[4] ProductoID reconstruido para las {df['ProductoID'].notna().sum()} filas con numero identificable")
    return df

In [24]:
# Normalizar Marca y Estado
def normalizar_marca(df):
    marcas_validas = {"Marca A", "Marca B", "Marca C", "Marca D"}
    invalidas = ~df["Marca"].isin(marcas_validas)
    print(f"[5] Marca invalida/faltante reemplazada por 'Sin marca': {invalidas.sum()}")
    df.loc[invalidas, "Marca"] = "Sin marca"
    return df


def normalizar_estado(df):
    estados_validos = {"Activo", "Descontinuado"}
    invalidos = ~df["Estado"].isin(estados_validos)
    print(f"[6] Estado invalido/faltante estandarizado a 'Sin dato': {invalidos.sum()}")
    df.loc[invalidos, "Estado"] = "Sin dato"
    return df

In [25]:
# Convertir Costo y Precio a numérico
def convertir_costo_precio(df):
    df["Costo_Unitario"] = pd.to_numeric(df["Costo_Unitario"], errors="coerce")
    df["Precio_Lista"] = pd.to_numeric(df["Precio_Lista"], errors="coerce")
    print("[7] Costo_Unitario y Precio_Lista convertidos a numerico")
    return df

In [26]:
# Detectar outliers (IQR) e imputar usando el ratio promedio costo/precio
def calcular_limites_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def imputar_costo_precio(df):
    costo, precio = df["Costo_Unitario"], df["Precio_Lista"]

    costo_invalido = costo.isna() | (costo <= 0)
    precio_invalido = precio.isna() | (precio <= 0)

    # outliers de costo via IQR, calculados solo sobre valores ya validos
    low, high = calcular_limites_iqr(costo[~costo_invalido])
    costo_invalido = costo_invalido | ((~costo_invalido) & ((costo < low) | (costo > high)))

    # ratio promedio costo/precio usando SOLO filas donde ambos son validos
    ambos_validos = (~costo_invalido) & (~precio_invalido)
    ratio_promedio = (costo[ambos_validos] / precio[ambos_validos]).mean()
    print(f"[8] Rango valido de Costo_Unitario (IQR): {low:,.0f} - {high:,.0f}")
    print(f"[9] Ratio promedio Costo/Precio calculado con {ambos_validos.sum()} registros validos: {ratio_promedio:.2%}")

    solo_costo_invalido = costo_invalido & (~precio_invalido)
    solo_precio_invalido = precio_invalido & (~costo_invalido)
    ambos_invalidos = costo_invalido & precio_invalido

    df.loc[solo_costo_invalido, "Costo_Unitario"] = (df.loc[solo_costo_invalido, "Precio_Lista"] * ratio_promedio).round()
    df.loc[solo_precio_invalido, "Precio_Lista"] = (df.loc[solo_precio_invalido, "Costo_Unitario"] / ratio_promedio).round()

    print(f"[10] Costo_Unitario imputado desde Precio_Lista: {solo_costo_invalido.sum()} filas")
    print(f"[11] Precio_Lista imputado desde Costo_Unitario: {solo_precio_invalido.sum()} filas")

    if ambos_invalidos.sum() > 0:
        # fallback (no ocurre en este dataset): promedio de la misma categoria
        for idx in df[ambos_invalidos].index:
            cat = df.loc[idx, "Categoria"]
            precio_cat = df.loc[(df["Categoria"] == cat) & (~precio_invalido), "Precio_Lista"].mean()
            df.loc[idx, "Precio_Lista"] = round(precio_cat)
            df.loc[idx, "Costo_Unitario"] = round(precio_cat * ratio_promedio)
        print(f"[12] ATENCION: {ambos_invalidos.sum()} filas con ambos invalidos -> imputadas con promedio de categoria")

    df["Costo_Unitario"] = df["Costo_Unitario"].astype(int)
    df["Precio_Lista"] = df["Precio_Lista"].astype(int)
    return df

In [27]:
# Validación final
def validar_resultado(df):
    assert len(df) == 120, f"Se esperaban 120 productos validos, hay {len(df)}"
    assert df["ProductoID"].is_unique, "Hay ProductoID duplicados"
    assert df["ProductoID"].notna().all(), "Quedaron ProductoID sin reconstruir"
    assert df["Categoria"].isin(CATEGORIAS_VALIDAS).all(), "Quedan categorias invalidas"
    assert (df["Costo_Unitario"] > 0).all() and (df["Precio_Lista"] > 0).all(), "Quedaron costos/precios <= 0"
    print("[13] Validacion final OK: 120 productos, ProductoID unicos, categorias validas, costos/precios positivos")

In [28]:
# Función principal
def main():
    df = cargar_datos(ARCHIVO_ENTRADA)
    df = eliminar_duplicados(df)
    df = reconstruir_producto_categoria_id(df)
    df = normalizar_marca(df)
    df = normalizar_estado(df)
    df = convertir_costo_precio(df)
    df = imputar_costo_precio(df)
    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.head(15).to_string())

[1] Duplicados eliminados: 2 (quedan 120 filas)
[2] Categoria reconstruida/corregida en 4 filas
[3] Producto reconstruido en 1 filas
[4] ProductoID reconstruido para las 120 filas con numero identificable
[5] Marca invalida/faltante reemplazada por 'Sin marca': 4
[6] Estado invalido/faltante estandarizado a 'Sin dato': 2
[7] Costo_Unitario y Precio_Lista convertidos a numerico
[8] Rango valido de Costo_Unitario (IQR): -86,750 - 479,250
[9] Ratio promedio Costo/Precio calculado con 117 registros validos: 64.24%
[10] Costo_Unitario imputado desde Precio_Lista: 2 filas
[11] Precio_Lista imputado desde Costo_Unitario: 1 filas
[13] Validacion final OK: 120 productos, ProductoID unicos, categorias validas, costos/precios positivos

Archivo limpio guardado en: /content/sample_data/03_productos_limpio.xlsx
   ProductoID               Producto   Categoria    Marca  Costo_Unitario  Precio_Lista         Estado
0   PROD-0001        Moda Producto 1        Moda  Marca D          347000        524454

### 02_Clientes

In [29]:
# Configuración
import pandas as pd
import re

ARCHIVO_ENTRADA = "/content/sample_data/02_clientes.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/02_clientes_limpio.xlsx"

SEGMENTOS_VALIDOS = {"B2B", "B2C", "Mayorista"}
ESTADOS_VALIDOS = {"Activo", "Inactivo"}

PATRON_ID = re.compile(r"^CLI-(\d{4})$")
PATRON_NOMBRE = re.compile(r"^\s*(Persona|Empresa)\s+([A-Za-zÀ-ÿ]+)\s+(\d+)\s*$", re.IGNORECASE)

# Correcciones puntuales de Ciudad que el title-case no resuelve bien (siglas)
CIUDADES_ESPECIALES = {"Cdmx": "CDMX"}


def cargar_datos(path):
    return pd.read_excel(path)

In [30]:
# Eliminar duplicados
def eliminar_duplicados(df):
    antes = len(df)
    df = df.drop_duplicates(subset=["Cliente", "País", "Ciudad", "Segmento", "Estado"]).reset_index(drop=True)
    print(f"[1] Duplicados eliminados: {antes - len(df)} (quedan {len(df)} filas)")
    return df

In [31]:
# Reconstruir ClienteID
def reconstruir_cliente_id(df):
    """
    El ClienteID, cuando tiene formato valido (CLI-####), es la fuente de verdad
    (coincide con el numero del nombre en 392/393 casos validados).
    Si es invalido o falta, se reconstruye extrayendo el numero desde 'Cliente'.
    """
    nuevos_id = []
    reconstruidos = 0

    for _, fila in df.iterrows():
        id_actual = str(fila["ClienteID"]).strip()
        m_id = PATRON_ID.match(id_actual.upper())

        if m_id:
            nuevos_id.append(f"CLI-{m_id.group(1)}")
        else:
            m_nombre = PATRON_NOMBRE.match(str(fila["Cliente"]).strip())
            if m_nombre:
                numero = int(m_nombre.group(3))
                nuevos_id.append(f"CLI-{numero:04d}")
                reconstruidos += 1
            else:
                nuevos_id.append(id_actual)  # no deberia ocurrir en este dataset

    df["ClienteID"] = nuevos_id
    print(f"[2] ClienteID reconstruido desde el nombre en {reconstruidos} filas")
    return df

In [32]:
# Normalizar columnas de texto
def normalizar_texto(df):
    df["Cliente"] = df["Cliente"].str.strip().str.title()
    df["País"] = df["País"].str.strip().str.capitalize()
    df["Ciudad"] = df["Ciudad"].str.strip().str.title().replace(CIUDADES_ESPECIALES)
    print("[3] Columnas de texto normalizadas: Cliente, País, Ciudad")
    return df

In [35]:
# Normalizar y reemplazar Segmento y Estado
def normalizar_segmento_estado(df):
    # Estado: primero normalizar mayusculas/minusculas antes de revisar invalidos
    df["Estado"] = df["Estado"].str.strip().str.capitalize()

    segmento_invalido = ~df["Segmento"].isin(SEGMENTOS_VALIDOS)
    estado_invalido = ~df["Estado"].isin(ESTADOS_VALIDOS)

    moda_segmento = df.loc[~segmento_invalido, "Segmento"].mode().iloc[0]
    moda_estado = df.loc[~estado_invalido, "Estado"].mode().iloc[0]

    print(f"[4] Segmento invalido/faltante reemplazado por moda global ('{moda_segmento}'): {segmento_invalido.sum()}")
    print(f"[5] Estado invalido/faltante reemplazado por moda global ('{moda_estado}'): {estado_invalido.sum()}")

    df.loc[segmento_invalido, "Segmento"] = moda_segmento
    df.loc[estado_invalido, "Estado"] = moda_estado
    return df

In [36]:
# Validación final
def validar_resultado(df):
    assert len(df) == 400, f"Se esperaban 400 clientes validos, hay {len(df)}"
    assert df["ClienteID"].is_unique, "Hay ClienteID duplicados"
    assert df["Segmento"].isin(SEGMENTOS_VALIDOS).all(), "Quedan segmentos invalidos"
    assert df["Estado"].isin(ESTADOS_VALIDOS).all(), "Quedan estados invalidos"
    print("[6] Validacion final OK: 400 clientes, ClienteID unicos, Segmento y Estado validos")

In [37]:
# Función principal
def main():
    df = cargar_datos(ARCHIVO_ENTRADA)
    df = eliminar_duplicados(df)
    df = reconstruir_cliente_id(df)
    df = normalizar_texto(df)
    df = normalizar_segmento_estado(df)
    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.head(15).to_string())

[1] Duplicados eliminados: 6 (quedan 400 filas)
[2] ClienteID reconstruido desde el nombre en 7 filas
[3] Columnas de texto normalizadas: Cliente, País, Ciudad
[4] Segmento invalido/faltante reemplazado por moda global ('B2C'): 8
[5] Estado invalido/faltante reemplazado por moda global ('Activo'): 3
[6] Validacion final OK: 400 clientes, ClienteID unicos, Segmento y Estado validos

Archivo limpio guardado en: /content/sample_data/02_clientes_limpio.xlsx
   ClienteID             Cliente       País        Ciudad   Segmento    Estado
0   CLI-0001     Persona Norte 1     México   Guadalajara        B2B    Activo
1   CLI-0002     Persona Smart 2   Colombia          Cali        B2C    Activo
2   CLI-0003    Empresa Global 3       Perú      Arequipa        B2B    Activo
3   CLI-0004     Empresa Andes 4       Perú      Arequipa        B2B    Activo
4   CLI-0005    Persona Global 5   Colombia          Cali        B2B  Inactivo
5   CLI-0006     Persona Norte 6   Colombia          Cali  Mayorista

### 01_Ventas

In [56]:
# Configuración
import pandas as pd
import numpy as np
import re

ARCHIVO_ENTRADA = "/content/sample_data/01_ventas.xlsx"
ARCHIVO_PRODUCTOS = "/content/sample_data/03_productos_limpio.xlsx"   # el que ya limpiamos antes
ARCHIVO_SALIDA = "/content/sample_data/01_ventas_limpio.xlsx"

TOTAL_VENTAS_VALIDAS = 1800

CANALES_VALIDOS = {"Marketplace", "Online", "Distribuidor", "Tienda"}
METODOS_VALIDOS = {"Tarjeta", "Efectivo", "Transferencia", "Crédito"}

PATRONES_ID = {
    "ClienteID": (re.compile(r"^([A-Za-z]+)-?(\d{4})$"), "CLI"),
    "ProductoID": (re.compile(r"^([A-Za-z]+)-?(\d{4})$"), "PROD"),
    "VendedorID": (re.compile(r"^([A-Za-z]+)-?(\d{3})$"), "VEN"),
    "SucursalID": (re.compile(r"^([A-Za-z]+)-?(\d{3})$"), "SUC"),
}


def cargar_datos():
    ventas = pd.read_excel(ARCHIVO_ENTRADA)
    productos = pd.read_excel(ARCHIVO_PRODUCTOS)
    return ventas, productos

In [57]:
# Eliminar duplicados
def eliminar_duplicados(df):
    antes = len(df)
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "VentaID"]).reset_index(drop=True)
    print(f"[1] Duplicados eliminados: {antes - len(df)} (quedan {len(df)} filas)")
    return df

In [58]:
# Reconstruir VentaID
def reconstruir_venta_id(df):
    """
    VentaID debe ser una permutacion unica de VTA-00001 a VTA-01800.
    Se recupera el numero de los IDs mal formateados (case/guion), y para
    los irrecuperables (NaN/NO_EXISTE) se asignan los numeros que faltan
    en la secuencia 1-1800 (se verifico que coinciden exactamente en cantidad).
    """
    patron_flexible = re.compile(r"VTA-?(\d{4,5})", re.IGNORECASE)

    numeros = []
    filas_irrecuperables = []
    for i, v in enumerate(df["VentaID"]):
        m = patron_flexible.search(str(v))
        if m:
            numeros.append(int(m.group(1)))
        else:
            numeros.append(None)
            filas_irrecuperables.append(i)

    usados = {n for n in numeros if n is not None}
    faltantes = sorted(set(range(1, TOTAL_VENTAS_VALIDAS + 1)) - usados)

    if len(faltantes) != len(filas_irrecuperables):
        raise ValueError(
            f"No coincide la cantidad de numeros faltantes ({len(faltantes)}) "
            f"con filas irrecuperables ({len(filas_irrecuperables)}); revisar antes de continuar."
        )

    for idx_fila, numero in zip(filas_irrecuperables, faltantes):
        numeros[idx_fila] = numero

    df["VentaID"] = [f"VTA-{n:05d}" for n in numeros]
    print(f"[2] VentaID reconstruido: {len(filas_irrecuperables)} filas irrecuperables completadas con la numeracion faltante")
    return df

In [59]:
# Fecha e IDs de referencia
def convertir_y_completar_fecha(df):
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    faltantes = df["Fecha"].isna().sum()
    mediana = df["Fecha"].median()
    df["Fecha"] = df["Fecha"].fillna(mediana)
    print(f"[3] Fecha convertida a datetime; {faltantes} valor(es) faltante(s) completado(s) con la mediana ({mediana.date()})")
    return df


def estandarizar_ids(df):
    """
    Normaliza formato (mayusculas + guion) cuando el ID es recuperable.
    Cuando es irrecuperable (NaN / NO_EXISTE / no matchea ningun patron),
    se marca explicitamente como 'Sin dato' para no inventar una referencia
    y no corromper el cruce con las tablas de dimension.
    """
    reporte = {}
    for columna, (patron, prefijo) in PATRONES_ID.items():
        irrecuperables = 0
        normalizados = 0
        nuevos = []

        for valor in df[columna]:
            s = str(valor).strip()
            m = patron.match(s.upper())
            if m and m.group(1).upper() == prefijo:
                nuevo_valor = f"{prefijo}-{m.group(2)}"
                if nuevo_valor != s:
                    normalizados += 1
                nuevos.append(nuevo_valor)
            else:
                nuevos.append("Sin dato")
                irrecuperables += 1

        df[columna] = nuevos
        reporte[columna] = (normalizados, irrecuperables)

    print("[4] IDs estandarizados (normalizados_formato, irrecuperables->'Sin dato'):")
    for col, (norm, irrec) in reporte.items():
        print(f"     {col}: {norm} normalizados, {irrec} irrecuperables")
    return df

In [60]:
# Canal, Método de Pago y Cantidad
def normalizar_canal_metodo_pago(df):
    """
    Los valores invalidos/faltantes se marcan con una etiqueta explicita
    ('Sin canal' / 'Sin método de pago'), en vez de la moda global, para
    mantener el mismo criterio usado en otras columnas categoricas del
    proyecto (ej. Nivel -> 'Sin nivel', Marca -> 'Sin marca').
    """
    df["Canal"] = df["Canal"].astype(str).str.strip().str.title()
    df["Metodo_Pago"] = df["Metodo_Pago"].astype(str).str.strip().str.title()

    canal_invalido = ~df["Canal"].isin(CANALES_VALIDOS)
    metodo_invalido = ~df["Metodo_Pago"].isin(METODOS_VALIDOS)

    print(f"[5] Canal invalido/faltante reemplazado por 'Sin canal': {canal_invalido.sum()}")
    print(f"[6] Metodo_Pago invalido/faltante reemplazado por 'Sin método de pago': {metodo_invalido.sum()}")

    df.loc[canal_invalido, "Canal"] = "Sin canal"
    df.loc[metodo_invalido, "Metodo_Pago"] = "Sin método de pago"
    return df


def calcular_limites_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def limpiar_cantidad(df):
    df["Cantidad"] = pd.to_numeric(df["Cantidad"], errors="coerce")

    invalido = df["Cantidad"].isna() | (df["Cantidad"] <= 0)
    low, high = calcular_limites_iqr(df.loc[~invalido, "Cantidad"])
    invalido = invalido | ((~invalido) & ((df["Cantidad"] < low) | (df["Cantidad"] > high)))

    mediana = df.loc[~invalido, "Cantidad"].median()
    print(f"[7] Cantidad invalida (texto/negativos/cero/outliers IQR) imputada con mediana ({mediana:.0f}): {invalido.sum()}")

    df.loc[invalido, "Cantidad"] = mediana
    df["Cantidad"] = df["Cantidad"].astype(int)
    return df

In [61]:
# Precio_Unitario desde el catálogo de productos
def reconstruir_precio_unitario(df, productos):
    """
    El precio correcto se obtiene del catalogo de productos (Precio_Lista),
    cruzando por ProductoID. Es la fuente de verdad: se sobrescribe el precio
    de ventas aunque el original pareciera valido, para garantizar consistencia
    con el catalogo oficial.
    Si el ProductoID no es resoluble ('Sin dato' o no existe en el catalogo),
    se usa el precio original de la venta si es numerico valido; si tampoco,
    se imputa con la mediana global de precios del catalogo.
    """
    precios_catalogo = productos.set_index("ProductoID")["Precio_Lista"]

    precio_desde_catalogo = df["ProductoID"].map(precios_catalogo)
    resuelto_por_catalogo = precio_desde_catalogo.notna()

    precio_original = pd.to_numeric(df["Precio_Unitario"], errors="coerce")
    mediana_catalogo = precios_catalogo.median()

    precio_final = precio_desde_catalogo.copy()
    usar_original = (~resuelto_por_catalogo) & precio_original.notna() & (precio_original > 0)
    precio_final[usar_original] = precio_original[usar_original]

    usar_mediana = (~resuelto_por_catalogo) & (~usar_original)
    precio_final[usar_mediana] = mediana_catalogo

    df["Precio_Unitario"] = precio_final.round().astype(int)

    print(f"[8] Precio_Unitario reconstruido desde catalogo de productos: {resuelto_por_catalogo.sum()} filas")
    print(f"     Usando precio original (ProductoID no resoluble): {usar_original.sum()} filas")
    print(f"     Usando mediana global del catalogo (sin ninguna referencia): {usar_mediana.sum()} filas")
    return df

In [62]:
# Descuento y Total_Venta
def parsear_descuento(valor):
    """
    Convierte texto/porcentaje a fraccion numerica.
    CORRECCION: un numero > 1 solo tiene sentido como porcentaje
    (ej. 1.5 -> 1.5%, 15 -> 15%), aunque no traiga el simbolo '%'.
    Por eso tambien se divide por 100 en ese caso, no solo cuando
    aparece el simbolo explicito.
    """
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip()
    tiene_signo_pct = "%" in s
    s_limpio = s.replace("%", "").strip()

    try:
        numero = float(s_limpio)
    except ValueError:
        return np.nan

    if tiene_signo_pct or numero > 1:
        return numero / 100
    return numero


def limpiar_descuento(df):
    """
    Convierte texto/porcentaje a fraccion numerica. Cualquier valor fuera
    de [0, 0.2] (o no interpretable) se considera invalido y pasa a 0,
    segun la regla explicita de la pauta.
    """
    parseado = df["Descuento"].apply(parsear_descuento)
    fuera_de_rango = parseado.isna() | (parseado < 0) | (parseado > 0.2)

    print(f"[9] Descuento fuera de rango [0, 0.2] o no interpretable -> 0: {fuera_de_rango.sum()}")

    parseado[fuera_de_rango] = 0
    df["Descuento"] = parseado
    return df


def recalcular_total_venta(df):
    df["Total_Venta"] = (df["Precio_Unitario"] * df["Cantidad"] * (1 - df["Descuento"])).round().astype(int)
    print("[10] Total_Venta recalculado: Precio_Unitario * Cantidad * (1 - Descuento)")
    return df

In [63]:
# Validación final
def validar_resultado(df):
    assert len(df) == TOTAL_VENTAS_VALIDAS, f"Se esperaban {TOTAL_VENTAS_VALIDAS} ventas validas, hay {len(df)}"
    assert df["VentaID"].is_unique, "Hay VentaID duplicados"
    assert df["Fecha"].isna().sum() == 0, "Quedaron fechas nulas"
    assert df["Descuento"].between(0, 0.2).all(), "Quedaron descuentos fuera de rango"
    assert (df["Cantidad"] > 0).all(), "Quedaron cantidades <= 0"
    assert (df["Precio_Unitario"] > 0).all(), "Quedaron precios <= 0"
    print(f"[11] Validacion final OK: {TOTAL_VENTAS_VALIDAS} ventas, VentaID unicos, sin nulos criticos")

In [66]:
# Función principal
def main():
    ventas, productos = cargar_datos()
    df = eliminar_duplicados(ventas)
    df = reconstruir_venta_id(df)
    df = convertir_y_completar_fecha(df)
    df = estandarizar_ids(df)
    df = normalizar_canal_metodo_pago(df)
    df = limpiar_cantidad(df)
    df = reconstruir_precio_unitario(df, productos)
    df = limpiar_descuento(df)
    df = recalcular_total_venta(df)
    validar_resultado(df)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nArchivo limpio guardado en: {ARCHIVO_SALIDA}")
    return df


if __name__ == "__main__":
    df_limpio = main()
    print(df_limpio.head(15).to_string())

[1] Duplicados eliminados: 27 (quedan 1800 filas)
[2] VentaID reconstruido: 12 filas irrecuperables completadas con la numeracion faltante
[3] Fecha convertida a datetime; 22 valor(es) faltante(s) completado(s) con la mediana (2024-08-13)
[4] IDs estandarizados (normalizados_formato, irrecuperables->'Sin dato'):
     ClienteID: 11 normalizados, 11 irrecuperables
     ProductoID: 6 normalizados, 6 irrecuperables
     VendedorID: 13 normalizados, 8 irrecuperables
     SucursalID: 11 normalizados, 11 irrecuperables
[5] Canal invalido/faltante reemplazado por 'Sin canal': 14
[6] Metodo_Pago invalido/faltante reemplazado por 'Sin método de pago': 14
[7] Cantidad invalida (texto/negativos/cero/outliers IQR) imputada con mediana (4): 20
[8] Precio_Unitario reconstruido desde catalogo de productos: 1794 filas
     Usando precio original (ProductoID no resoluble): 6 filas
     Usando mediana global del catalogo (sin ninguna referencia): 0 filas
[9] Descuento fuera de rango [0, 0.2] o no interpr

## Carga y unificación de tablas

In [67]:
# Configuración y carga
import pandas as pd

RUTA_VENTAS = "/content/sample_data/01_ventas_limpio.xlsx"
RUTA_CLIENTES = "/content/sample_data/02_clientes_limpio.xlsx"
RUTA_PRODUCTOS = "/content/sample_data/03_productos_limpio.xlsx"
RUTA_VENDEDORES = "/content/sample_data/04_vendedores_limpio.xlsx"
RUTA_SUCURSALES = "/content/sample_data/05_sucursales_limpio.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/base_analitica_final.xlsx"


def cargar_tablas():
    ventas = pd.read_excel(RUTA_VENTAS)
    clientes = pd.read_excel(RUTA_CLIENTES)
    productos = pd.read_excel(RUTA_PRODUCTOS)
    vendedores = pd.read_excel(RUTA_VENDEDORES)
    sucursales = pd.read_excel(RUTA_SUCURSALES)
    return ventas, clientes, productos, vendedores, sucursales

In [68]:
# Limpiar espacios en nombres de columnas
def limpiar_nombres_columnas(*tablas):
    """Quita espacios accidentales en los encabezados de todas las tablas."""
    limpias = []
    for df in tablas:
        df.columns = df.columns.str.strip()
        limpias.append(df)
    print("[1] Nombres de columnas limpiados (sin espacios) en las 5 tablas")
    return limpias

In [69]:
# Renombrar columnas ambiguas (antes del merge)
def renombrar_columnas_ambiguas(clientes, productos, vendedores, sucursales):
    """
    Varias tablas comparten nombres de columna con significados distintos
    (Estado, Ciudad, Pais/País, SucursalID). Se renombran ANTES del merge
    para que no se generen sufijos automaticos (_x, _y) sin significado.
    """
    clientes = clientes.rename(columns={
        "Estado": "Estado_Cliente",
        "País": "Pais_Cliente",
        "Ciudad": "Ciudad_Cliente",
    })
    productos = productos.rename(columns={"Estado": "Estado_Producto"})
    vendedores = vendedores.rename(columns={
        "Estado": "Estado_Vendedor",
        "SucursalID": "SucursalID_Vendedor",  # la sucursal ASIGNADA al vendedor, no la de la venta
    })
    sucursales = sucursales.rename(columns={
        "Estado": "Estado_Sucursal",
        "Pais": "Pais_Sucursal",
        "Ciudad": "Ciudad_Sucursal",
    })
    print("[2] Columnas ambiguas renombradas con su tabla de origen")
    return clientes, productos, vendedores, sucursales

In [70]:
# Unir las tablas
def unir_tablas(ventas, clientes, productos, vendedores, sucursales):
    """
    Ventas es la tabla ancla: se usa 'left' en todos los merge para
    mantener las 1800 ventas aunque falten datos relacionados
    (ej. ClienteID = 'Sin dato').
    IMPORTANTE: el SucursalID de Ventas NUNCA se sobrescribe; el de
    Vendedores quedo renombrado a SucursalID_Vendedor.
    """
    filas_iniciales = len(ventas)

    base = ventas.merge(clientes, on="ClienteID", how="left")
    base = base.merge(productos, on="ProductoID", how="left")
    base = base.merge(vendedores, on="VendedorID", how="left")
    base = base.merge(sucursales, on="SucursalID", how="left")

    print(f"[3] Tablas unidas: {filas_iniciales} filas antes -> {len(base)} filas despues (deben coincidir)")
    return base

In [71]:
# Validación final
def validar_resultado(base, filas_esperadas):
    assert len(base) == filas_esperadas, f"Se esperaban {filas_esperadas} filas, hay {len(base)}"
    assert base["SucursalID"].notna().all(), "Se perdio el SucursalID original de ventas"
    columnas_duplicadas = base.columns[base.columns.duplicated()].tolist()
    assert not columnas_duplicadas, f"Hay columnas duplicadas: {columnas_duplicadas}"
    print(f"[4] Validacion final OK: {len(base)} filas, sin columnas duplicadas, SucursalID de ventas preservado")

In [72]:
# Función principal
def main():
    ventas, clientes, productos, vendedores, sucursales = cargar_tablas()
    ventas, clientes, productos, vendedores, sucursales = limpiar_nombres_columnas(
        ventas, clientes, productos, vendedores, sucursales
    )
    clientes, productos, vendedores, sucursales = renombrar_columnas_ambiguas(
        clientes, productos, vendedores, sucursales
    )
    base = unir_tablas(ventas, clientes, productos, vendedores, sucursales)
    validar_resultado(base, filas_esperadas=len(ventas))

    base.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nBase analitica final guardada en: {ARCHIVO_SALIDA}")
    print(f"Dimensiones finales: {base.shape}")
    return base


if __name__ == "__main__":
    base_final = main()
    print(base_final.columns.tolist())
    print(base_final.head(5).to_string())

[1] Nombres de columnas limpiados (sin espacios) en las 5 tablas
[2] Columnas ambiguas renombradas con su tabla de origen
[3] Tablas unidas: 1800 filas antes -> 1800 filas despues (deben coincidir)
[4] Validacion final OK: 1800 filas, sin columnas duplicadas, SucursalID de ventas preservado

Base analitica final guardada en: /content/sample_data/base_analitica_final.xlsx
Dimensiones finales: (1800, 32)
['VentaID', 'Fecha', 'ClienteID', 'ProductoID', 'VendedorID', 'SucursalID', 'Canal', 'Metodo_Pago', 'Cantidad', 'Precio_Unitario', 'Descuento', 'Total_Venta', 'Cliente', 'Pais_Cliente', 'Ciudad_Cliente', 'Segmento', 'Estado_Cliente', 'Producto', 'Categoria', 'Marca', 'Costo_Unitario', 'Precio_Lista', 'Estado_Producto', 'Vendedor', 'SucursalID_Vendedor', 'Nivel', 'Estado_Vendedor', 'Sucursal', 'Pais_Sucursal', 'Ciudad_Sucursal', 'Zona', 'Estado_Sucursal']
     VentaID      Fecha ClienteID ProductoID VendedorID SucursalID         Canal    Metodo_Pago  Cantidad  Precio_Unitario  Descuento  

## Transformación de datos y creación de métricas comerciales

In [73]:
# Configuración
import pandas as pd

ARCHIVO_ENTRADA = "/content/sample_data/base_analitica_final.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/base_analitica_transformada.xlsx"

NOMBRES_MES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
NOMBRES_DIA = {
    0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo",
}


def cargar_datos():
    return pd.read_excel(ARCHIVO_ENTRADA)

In [74]:
# Fecha y columnas de tiempo
def convertir_fecha(df):
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    print("[1] Fecha confirmada/convertida a datetime")
    return df


def crear_columnas_tiempo(df):
    df["Año"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.month
    df["NombreMes"] = df["Mes"].map(NOMBRES_MES)
    df["Trimestre"] = df["Fecha"].dt.quarter
    df["DiaSemana"] = df["Fecha"].dt.dayofweek.map(NOMBRES_DIA)
    df["EsFinDeSemana"] = df["Fecha"].dt.dayofweek.isin([5, 6])
    print("[2] Columnas de tiempo creadas: Año, Mes, NombreMes, Trimestre, DiaSemana, EsFinDeSemana")
    return df

In [75]:
# Ventas brutas y netas
def calcular_ventas_brutas_netas(df):
    """
    Venta_Bruta: sin descuento. Venta_Neta: con descuento aplicado
    (debe coincidir con Total_Venta, calculado en la etapa anterior;
    se deja como verificacion cruzada).
    """
    df["Venta_Bruta"] = df["Precio_Unitario"] * df["Cantidad"]
    df["Venta_Neta"] = df["Venta_Bruta"] * (1 - df["Descuento"])

    diferencia_maxima = (df["Venta_Neta"].round() - df["Total_Venta"]).abs().max()
    print(f"[3] Venta_Bruta y Venta_Neta calculadas (diferencia maxima vs Total_Venta: {diferencia_maxima})")
    return df

In [76]:
# Costos, utilidad y margen
def calcular_costos_utilidad_margen(df):
    """
    Costo_Total = Costo_Unitario * Cantidad.
    Para las ventas cuyo ProductoID quedo como 'Sin dato' (sin costo conocido),
    se imputa Costo_Unitario usando la razon promedio Costo/Precio calculada
    con el resto de la base (misma logica aplicada en 03_productos), para
    mantener consistencia numerica en el 100% de la base.
    """
    ratio_valido = (df["Costo_Unitario"] / df["Precio_Lista"]).dropna()
    ratio_promedio = ratio_valido.mean()

    sin_costo = df["Costo_Unitario"].isna()
    print(f"[4] Ventas sin Costo_Unitario conocido: {sin_costo.sum()} -> imputadas con ratio promedio ({ratio_promedio:.2%}) sobre Precio_Unitario")

    df.loc[sin_costo, "Costo_Unitario"] = df.loc[sin_costo, "Precio_Unitario"] * ratio_promedio

    df["Costo_Total"] = df["Costo_Unitario"] * df["Cantidad"]
    df["Utilidad"] = df["Venta_Neta"] - df["Costo_Total"]
    df["Margen_%"] = (df["Utilidad"] / df["Venta_Neta"]) * 100

    print("[5] Costo_Total, Utilidad y Margen_% calculados para el 100% de las filas")
    return df

In [77]:
# Redondeo y consistencia de tipos
def redondear_y_estandarizar_tipos(df):
    columnas_monetarias = ["Venta_Bruta", "Venta_Neta", "Costo_Unitario", "Costo_Total", "Utilidad"]
    for col in columnas_monetarias:
        df[col] = df[col].round(0).astype(int)

    df["Margen_%"] = df["Margen_%"].round(2)
    df["Descuento"] = df["Descuento"].round(2)

    print(f"[6] Metricas monetarias redondeadas a enteros: {columnas_monetarias}")
    print("[7] Margen_% y Descuento redondeados a 2 decimales")
    return df

In [78]:
# Validación final
def validar_resultado(df, filas_esperadas):
    assert len(df) == filas_esperadas, f"Se esperaban {filas_esperadas} filas, hay {len(df)}"
    assert df["Costo_Total"].isna().sum() == 0, "Quedaron Costo_Total vacios"
    assert df["Utilidad"].isna().sum() == 0, "Quedaron Utilidad vacios"
    assert df["Margen_%"].isna().sum() == 0, "Quedaron Margen_% vacios"
    print(f"[8] Validacion final OK: {len(df)} filas, metricas comerciales completas para el 100% de la base")

In [79]:
# Función principal
def main():
    df = cargar_datos()
    filas_iniciales = len(df)

    df = convertir_fecha(df)
    df = crear_columnas_tiempo(df)
    df = calcular_ventas_brutas_netas(df)
    df = calcular_costos_utilidad_margen(df)
    df = redondear_y_estandarizar_tipos(df)
    validar_resultado(df, filas_iniciales)

    df.to_excel(ARCHIVO_SALIDA, index=False)
    print(f"\nBase transformada guardada en: {ARCHIVO_SALIDA}")
    print(f"Dimensiones finales: {df.shape}")
    return df


if __name__ == "__main__":
    df_transformado = main()
    print(df_transformado.head(10).to_string())

[1] Fecha confirmada/convertida a datetime
[2] Columnas de tiempo creadas: Año, Mes, NombreMes, Trimestre, DiaSemana, EsFinDeSemana
[3] Venta_Bruta y Venta_Neta calculadas (diferencia maxima vs Total_Venta: 0.0)
[4] Ventas sin Costo_Unitario conocido: 6 -> imputadas con ratio promedio (64.20%) sobre Precio_Unitario
[5] Costo_Total, Utilidad y Margen_% calculados para el 100% de las filas
[6] Metricas monetarias redondeadas a enteros: ['Venta_Bruta', 'Venta_Neta', 'Costo_Unitario', 'Costo_Total', 'Utilidad']
[7] Margen_% y Descuento redondeados a 2 decimales
[8] Validacion final OK: 1800 filas, metricas comerciales completas para el 100% de la base

Base transformada guardada en: /content/sample_data/base_analitica_transformada.xlsx
Dimensiones finales: (1800, 43)
     VentaID      Fecha ClienteID ProductoID VendedorID SucursalID         Canal    Metodo_Pago  Cantidad  Precio_Unitario  Descuento  Total_Venta               Cliente Pais_Cliente Ciudad_Cliente   Segmento Estado_Cliente    

## Generación de KPIs principales

In [80]:
# Configuración y carga
import pandas as pd

ARCHIVO_ENTRADA = "/content/sample_data/base_analitica_transformada.xlsx"


def cargar_base_transformada():
    return pd.read_excel(ARCHIVO_ENTRADA)

In [81]:
# Limpiar nombres de columnas
def limpiar_nombres_columnas(df):
    df.columns = df.columns.str.strip()
    print("[1] Nombres de columnas limpiados")
    return df

In [82]:
# Calcular los KPIs
def calcular_kpis(df):
    kpis = {}

    kpis["Ventas_Totales"] = df["Venta_Neta"].sum()
    kpis["Utilidad_Total"] = df["Utilidad"].sum()

    # Dos formas validas de "margen promedio": simple y ponderado
    kpis["Margen_Promedio_Simple_%"] = df["Margen_%"].mean()
    kpis["Margen_Promedio_Ponderado_%"] = (kpis["Utilidad_Total"] / kpis["Ventas_Totales"]) * 100

    kpis["Cantidad_Vendida"] = df["Cantidad"].sum()
    kpis["Numero_de_Ventas"] = df["VentaID"].nunique()
    kpis["Ticket_Promedio"] = kpis["Ventas_Totales"] / kpis["Numero_de_Ventas"]
    kpis["Descuento_Promedio_%"] = df["Descuento"].mean() * 100

    print("[2-9] KPIs calculados: Ventas Totales, Utilidad Total, Margen Promedio (simple y ponderado),")
    print("       Cantidad Vendida, Número de Ventas, Ticket Promedio, Descuento Promedio")
    return kpis

In [83]:
# Redondear
def redondear_kpis(kpis):
    kpis_redondeados = {}
    for nombre, valor in kpis.items():
        if "%" in nombre or "Ticket" in nombre:
            kpis_redondeados[nombre] = round(valor, 2)
        else:
            kpis_redondeados[nombre] = round(valor)
    return kpis_redondeados

In [84]:
# Mostrar resultados
def mostrar_resultados(kpis):
    print("\n" + "=" * 45)
    print("KPIs PRINCIPALES - SISTEMA DE ANALISIS COMERCIAL")
    print("=" * 45)
    print(f"Ventas Totales:              $ {kpis['Ventas_Totales']:,.0f}")
    print(f"Utilidad Total:              $ {kpis['Utilidad_Total']:,.0f}")
    print(f"Margen Promedio (simple):      {kpis['Margen_Promedio_Simple_%']:.2f} %")
    print(f"Margen Promedio (ponderado):   {kpis['Margen_Promedio_Ponderado_%']:.2f} %")
    print(f"Cantidad Vendida:            {kpis['Cantidad_Vendida']:,.0f} unidades")
    print(f"Número de Ventas:            {kpis['Numero_de_Ventas']:,.0f}")
    print(f"Ticket Promedio:             $ {kpis['Ticket_Promedio']:,.2f}")
    print(f"Descuento Promedio:            {kpis['Descuento_Promedio_%']:.2f} %")
    print("=" * 45)

In [85]:
# Función principal
def main():
    df = cargar_base_transformada()
    df = limpiar_nombres_columnas(df)
    kpis = calcular_kpis(df)
    kpis = redondear_kpis(kpis)
    mostrar_resultados(kpis)
    return kpis


if __name__ == "__main__":
    kpis_finales = main()

[1] Nombres de columnas limpiados
[2-9] KPIs calculados: Ventas Totales, Utilidad Total, Margen Promedio (simple y ponderado),
       Cantidad Vendida, Número de Ventas, Ticket Promedio, Descuento Promedio

KPIs PRINCIPALES - SISTEMA DE ANALISIS COMERCIAL
Ventas Totales:              $ 2,239,416,568
Utilidad Total:              $ 674,373,176
Margen Promedio (simple):      29.50 %
Margen Promedio (ponderado):   30.11 %
Cantidad Vendida:            8,073 unidades
Número de Ventas:            1,800
Ticket Promedio:             $ 1,244,120.32
Descuento Promedio:            8.30 %


## Creación de tablas resumen para análisis

In [86]:
# Configuración
import pandas as pd

ARCHIVO_ENTRADA = "/content/sample_data/base_analitica_transformada.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/tablas_resumen.xlsx"


def cargar_datos():
    return pd.read_excel(ARCHIVO_ENTRADA)

In [87]:
# Ventas por mes / categoría / país / canal
def ventas_por_mes(df):
    tabla = (
        df.groupby(["Año", "Mes", "NombreMes"], as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values(["Año", "Mes"])
        .reset_index(drop=True)
    )
    return tabla


def ventas_por_categoria(df):
    tabla = (
        df.groupby("Categoria", as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"),
             Cantidad_Vendida=("Cantidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values("Ventas", ascending=False)
        .reset_index(drop=True)
    )
    return tabla


def ventas_por_pais(df):
    """Se usa el pais de la SUCURSAL (donde ocurrio la venta), no el del cliente."""
    tabla = (
        df.groupby("Pais_Sucursal", as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values("Ventas", ascending=False)
        .reset_index(drop=True)
    )
    return tabla


def ventas_por_canal(df):
    tabla = (
        df.groupby("Canal", as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values("Ventas", ascending=False)
        .reset_index(drop=True)
    )
    return tabla

In [88]:
# Top 10 productos / vendedores y utilidad por sucursal
def top10_productos(df):
    tabla = (
        df.groupby(["ProductoID", "Producto"], as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Cantidad_Vendida=("Cantidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values("Ventas", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    return tabla


def top10_vendedores(df):
    tabla = (
        df.groupby(["VendedorID", "Vendedor"], as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"), Numero_Ventas=("VentaID", "count"))
        .sort_values("Ventas", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    return tabla


def utilidad_por_sucursal(df):
    tabla = (
        df.groupby(["SucursalID", "Sucursal"], as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"), Utilidad=("Utilidad", "sum"), Numero_Ventas=("VentaID", "count"))
    )
    tabla["Margen_%"] = (tabla["Utilidad"] / tabla["Ventas"] * 100).round(2)
    tabla = tabla.sort_values("Utilidad", ascending=False).reset_index(drop=True)
    return tabla

In [89]:
# Redondeo, validación y función principal
def redondear_montos(tabla):
    for col in tabla.columns:
        if col in ("Ventas", "Utilidad"):
            tabla[col] = tabla[col].round(0).astype(int)
    return tabla


def validar_resultados(tablas):
    print("\n" + "=" * 50)
    print("VALIDACION DE TABLAS RESUMEN")
    print("=" * 50)
    for nombre, tabla in tablas.items():
        print(f"\n--- {nombre} ({len(tabla)} filas) ---")
        print(tabla.head().to_string(index=False))


def main():
    df = cargar_datos()

    tablas = {
        "Ventas_por_Mes": redondear_montos(ventas_por_mes(df)),
        "Ventas_por_Categoria": redondear_montos(ventas_por_categoria(df)),
        "Ventas_por_Pais": redondear_montos(ventas_por_pais(df)),
        "Ventas_por_Canal": redondear_montos(ventas_por_canal(df)),
        "Top10_Productos": redondear_montos(top10_productos(df)),
        "Top10_Vendedores": redondear_montos(top10_vendedores(df)),
        "Utilidad_por_Sucursal": redondear_montos(utilidad_por_sucursal(df)),
    }

    validar_resultados(tablas)

    with pd.ExcelWriter(ARCHIVO_SALIDA) as writer:
        for nombre, tabla in tablas.items():
            tabla.to_excel(writer, sheet_name=nombre[:31], index=False)

    print(f"\nTodas las tablas resumen guardadas en: {ARCHIVO_SALIDA} (una hoja por tabla)")
    return tablas


if __name__ == "__main__":
    tablas_resumen = main()


VALIDACION DE TABLAS RESUMEN

--- Ventas_por_Mes (15 filas) ---
 Año  Mes NombreMes    Ventas  Utilidad  Numero_Ventas
2024    1     Enero 133793270  40405438            107
2024    2   Febrero 154034387  46395933            115
2024    3     Marzo 173174533  50552875            125
2024    4     Abril 155215012  48155379            123
2024    5      Mayo 152764064  44860160            119

--- Ventas_por_Categoria (6 filas) ---
 Categoria    Ventas  Utilidad  Cantidad_Vendida  Numero_Ventas
Tecnología 509986008 151284872              1742            365
     Hogar 437405916 140993916              1600            356
      Moda 405548441 117884773              1467            343
   Oficina 339651705  93760705              1323            300
   Belleza 313690743 101137743              1137            260

--- Ventas_por_Pais (5 filas) ---
Pais_Sucursal    Ventas  Utilidad  Numero_Ventas
        Chile 617813933 186816520            504
       México 497638994 148429426            374

## Cumplimiento de metas

In [90]:
# Configuración y carga
import pandas as pd

ARCHIVO_VENTAS = "/content/sample_data/base_analitica_transformada.xlsx"
ARCHIVO_METAS = "/content/sample_data/06_metas_limpio.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/cumplimiento_metas.xlsx"


def cargar_datos():
    ventas = pd.read_excel(ARCHIVO_VENTAS)
    metas = pd.read_excel(ARCHIVO_METAS)
    return ventas, metas

In [91]:
# Agregar ventas reales y preparar metas
def agregar_ventas_reales_por_sucursal_mes(ventas):
    """
    Excluye las ventas con SucursalID = 'Sin dato' (irrecuperable en la limpieza):
    no se pueden atribuir a ninguna sucursal, por lo que no corresponde
    incluirlas en el cumplimiento de metas de una sucursal especifica.
    """
    ventas_validas = ventas[ventas["SucursalID"] != "Sin dato"].copy()
    excluidas = len(ventas) - len(ventas_validas)

    reales = (
        ventas_validas.groupby(["SucursalID", "Año", "Mes"], as_index=False)
        .agg(Ventas_Reales=("Venta_Neta", "sum"), Unidades_Reales=("Cantidad", "sum"))
    )
    print(f"[1] Ventas reales agregadas por Sucursal/Año/Mes ({excluidas} ventas sin sucursal conocida excluidas)")
    return reales


def preparar_metas(metas):
    metas = metas.copy()
    metas["Fecha"] = pd.to_datetime(metas["Fecha"])
    metas["Año"] = metas["Fecha"].dt.year
    metas["Mes"] = metas["Fecha"].dt.month
    print("[2] Metas preparadas con columnas Año y Mes para el cruce")
    return metas

In [92]:
# Cruzar y calcular cumplimiento
def cruzar_metas_con_reales(metas, reales):
    """
    Metas es la tabla ancla (left join): asi se ve el cumplimiento de
    TODOS los meses/sucursales con meta definida, incluso si ese mes
    no hubo ninguna venta registrada (quedaria en 0, lo cual es real).
    """
    comparativa = metas.merge(reales, on=["SucursalID", "Año", "Mes"], how="left")
    comparativa["Ventas_Reales"] = comparativa["Ventas_Reales"].fillna(0)
    comparativa["Unidades_Reales"] = comparativa["Unidades_Reales"].fillna(0)

    sin_ventas = comparativa["Ventas_Reales"].eq(0).sum()
    print(f"[3] Metas cruzadas con ventas reales ({sin_ventas} meses/sucursal sin ninguna venta registrada)")
    return comparativa


def calcular_cumplimiento(comparativa):
    comparativa["Cumplimiento_Ventas_%"] = (
        comparativa["Ventas_Reales"] / comparativa["Meta_Ventas"] * 100
    ).round(2)
    comparativa["Cumplimiento_Unidades_%"] = (
        comparativa["Unidades_Reales"] / comparativa["Meta_Unidades"] * 100
    ).round(2)
    comparativa["Meta_Ventas_Cumplida"] = comparativa["Cumplimiento_Ventas_%"] >= 100
    comparativa["Meta_Unidades_Cumplida"] = comparativa["Cumplimiento_Unidades_%"] >= 100

    print("[4] Cumplimiento_Ventas_%, Cumplimiento_Unidades_% y banderas de cumplimiento calculadas")
    return comparativa


def redondear_montos(comparativa):
    comparativa["Ventas_Reales"] = comparativa["Ventas_Reales"].round(0).astype(int)
    comparativa["Unidades_Reales"] = comparativa["Unidades_Reales"].round(0).astype(int)
    return comparativa

In [93]:
# Resumen por sucursal, validación y guardado
def resumen_por_sucursal(comparativa):
    resumen = (
        comparativa.groupby("SucursalID", as_index=False)
        .agg(
            Meta_Ventas_Total=("Meta_Ventas", "sum"),
            Ventas_Reales_Total=("Ventas_Reales", "sum"),
            Meta_Unidades_Total=("Meta_Unidades", "sum"),
            Unidades_Reales_Total=("Unidades_Reales", "sum"),
        )
    )
    resumen["Cumplimiento_Ventas_%"] = (resumen["Ventas_Reales_Total"] / resumen["Meta_Ventas_Total"] * 100).round(2)
    resumen["Cumplimiento_Unidades_%"] = (resumen["Unidades_Reales_Total"] / resumen["Meta_Unidades_Total"] * 100).round(2)
    resumen = resumen.sort_values("Cumplimiento_Ventas_%", ascending=False).reset_index(drop=True)
    return resumen


def validar_y_mostrar(comparativa, resumen):
    assert comparativa["Ventas_Reales"].isna().sum() == 0, "Quedaron ventas reales sin completar"
    assert comparativa["Cumplimiento_Ventas_%"].isna().sum() == 0, "Quedaron cumplimientos sin calcular"

    print(comparativa.head().to_string(index=False))
    print(resumen.to_string(index=False))


def main():
    ventas, metas = cargar_datos()

    reales = agregar_ventas_reales_por_sucursal_mes(ventas)
    metas = preparar_metas(metas)
    comparativa = cruzar_metas_con_reales(metas, reales)
    comparativa = calcular_cumplimiento(comparativa)
    comparativa = redondear_montos(comparativa)

    resumen = resumen_por_sucursal(comparativa)
    validar_y_mostrar(comparativa, resumen)

    with pd.ExcelWriter(ARCHIVO_SALIDA) as writer:
        comparativa.to_excel(writer, sheet_name="Detalle_Mensual", index=False)
        resumen.to_excel(writer, sheet_name="Resumen_por_Sucursal", index=False)

    print(f"\nArchivo guardado en: {ARCHIVO_SALIDA}")
    return comparativa, resumen


if __name__ == "__main__":
    comparativa_final, resumen_final = main()

[1] Ventas reales agregadas por Sucursal/Año/Mes (11 ventas sin sucursal conocida excluidas)
[2] Metas preparadas con columnas Año y Mes para el cruce
[3] Metas cruzadas con ventas reales (15 meses/sucursal sin ninguna venta registrada)
[4] Cumplimiento_Ventas_%, Cumplimiento_Unidades_% y banderas de cumplimiento calculadas
SucursalID      Fecha  Meta_Ventas  Meta_Unidades  Año  Mes  Ventas_Reales  Unidades_Reales  Cumplimiento_Ventas_%  Cumplimiento_Unidades_%  Meta_Ventas_Cumplida  Meta_Unidades_Cumplida
   SUC-001 2024-01-01      6774353             38 2024    1        7513021               33                 110.90                    86.84                  True                   False
   SUC-001 2024-02-01      3579350             14 2024    2        3907617               13                 109.17                    92.86                  True                   False
   SUC-001 2024-03-01      5412798             23 2024    3        5003057               21                  92.43  

## Visualizaciones con Matplotlib

In [94]:
# Configuración y formateadores de etiquetas
import pandas as pd
import matplotlib.pyplot as plt
import os

ARCHIVO_BASE = "/content/sample_data/base_analitica_transformada.xlsx"
ARCHIVO_TABLAS = "/content/sample_data/tablas_resumen.xlsx"
ARCHIVO_METAS = "/content/sample_data/cumplimiento_metas.xlsx"
CARPETA_GRAFICOS = "/content/sample_data/graficos"


def crear_carpeta_graficos():
    os.makedirs(CARPETA_GRAFICOS, exist_ok=True)
    print(f"[1] Carpeta de graficos lista en: {CARPETA_GRAFICOS}")


def cargar_datos():
    base = pd.read_excel(ARCHIVO_BASE)
    ventas_por_categoria = pd.read_excel(ARCHIVO_TABLAS, sheet_name="Ventas_por_Categoria")
    top10_productos = pd.read_excel(ARCHIVO_TABLAS, sheet_name="Top10_Productos")
    top10_vendedores = pd.read_excel(ARCHIVO_TABLAS, sheet_name="Top10_Vendedores")
    ventas_por_canal = pd.read_excel(ARCHIVO_TABLAS, sheet_name="Ventas_por_Canal")
    utilidad_por_sucursal = pd.read_excel(ARCHIVO_TABLAS, sheet_name="Utilidad_por_Sucursal")
    resumen_metas = pd.read_excel(ARCHIVO_METAS, sheet_name="Resumen_por_Sucursal")
    return base, ventas_por_categoria, top10_productos, top10_vendedores, ventas_por_canal, utilidad_por_sucursal, resumen_metas


def guardar_grafico(nombre_archivo):
    ruta = os.path.join(CARPETA_GRAFICOS, nombre_archivo)
    plt.tight_layout()
    plt.savefig(ruta, dpi=150)
    plt.close()
    print(f"     -> Guardado: {ruta}")


def formato_monto(valor):
    """Convierte a millones para que la etiqueta sea legible (ej: 173200000 -> '$173.2M')."""
    return f"${valor/1_000_000:,.1f}M"


def formato_unidades(valor):
    return f"{valor:,.0f}"

In [95]:
# Gráfico de líneas (etiquetas en cada intersección) y ventas por categoría
def grafico_ventas_por_mes_por_anio(base):
    tabla = (
        base.groupby(["Año", "Mes"], as_index=False)
        .agg(Ventas=("Venta_Neta", "sum"))
    )

    plt.figure(figsize=(11, 7))
    for anio in sorted(tabla["Año"].unique()):
        datos_anio = tabla[tabla["Año"] == anio].sort_values("Mes")
        plt.plot(datos_anio["Mes"], datos_anio["Ventas"], marker="o", label=str(anio))

        # Etiqueta de monto en cada interseccion (punto)
        for x, y in zip(datos_anio["Mes"], datos_anio["Ventas"]):
            plt.annotate(
                formato_monto(y), (x, y),
                textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=8,
            )

    plt.title("Ventas por Mes, separado por Año", fontsize=14, fontweight="bold")
    plt.xlabel("Mes")
    plt.ylabel("Ventas ($)")
    plt.xticks(range(1, 13))
    plt.margins(y=0.15)  # espacio extra arriba para que las etiquetas no se corten
    plt.legend(title="Año")
    plt.grid(alpha=0.3)
    guardar_grafico("01_ventas_por_mes_por_anio.png")


def grafico_ventas_por_categoria(tabla):
    tabla = tabla.sort_values("Ventas", ascending=False)
    fig, ax = plt.subplots(figsize=(9, 6))
    barras = ax.bar(tabla["Categoria"], tabla["Ventas"], color="steelblue")

    ax.bar_label(barras, labels=[formato_monto(v) for v in tabla["Ventas"]], padding=3, fontsize=9)

    ax.set_title("Ventas por Categoría", fontsize=14, fontweight="bold")
    ax.set_xlabel("Categoría")
    ax.set_ylabel("Ventas ($)")
    ax.margins(y=0.12)
    plt.xticks(rotation=30, ha="right")
    ax.grid(axis="y", alpha=0.3)
    guardar_grafico("02_ventas_por_categoria.png")

In [96]:
# Top 10 productos, vendedores, canal y utilidad por sucursal
def grafico_top10_productos(tabla):
    tabla = tabla.sort_values("Ventas", ascending=True)
    fig, ax = plt.subplots(figsize=(11, 7))
    barras = ax.barh(tabla["Producto"], tabla["Ventas"], color="darkorange")

    ax.bar_label(barras, labels=[formato_monto(v) for v in tabla["Ventas"]], padding=3, fontsize=9)

    ax.set_title("Top 10 Productos por Ventas", fontsize=14, fontweight="bold")
    ax.set_xlabel("Ventas ($)")
    ax.set_ylabel("Producto")
    ax.margins(x=0.15)
    ax.grid(axis="x", alpha=0.3)
    guardar_grafico("03_top10_productos.png")


def grafico_top10_vendedores(tabla):
    tabla = tabla.sort_values("Ventas", ascending=True)
    fig, ax = plt.subplots(figsize=(11, 7))
    barras = ax.barh(tabla["Vendedor"], tabla["Ventas"], color="seagreen")

    ax.bar_label(barras, labels=[formato_monto(v) for v in tabla["Ventas"]], padding=3, fontsize=9)

    ax.set_title("Top 10 Vendedores por Ventas", fontsize=14, fontweight="bold")
    ax.set_xlabel("Ventas ($)")
    ax.set_ylabel("Vendedor")
    ax.margins(x=0.15)
    ax.grid(axis="x", alpha=0.3)
    guardar_grafico("04_top10_vendedores.png")


def grafico_ventas_por_canal(tabla):
    tabla = tabla.sort_values("Ventas", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 6))
    barras = ax.bar(tabla["Canal"], tabla["Ventas"], color="mediumpurple")

    ax.bar_label(barras, labels=[formato_monto(v) for v in tabla["Ventas"]], padding=3, fontsize=9)

    ax.set_title("Ventas por Canal", fontsize=14, fontweight="bold")
    ax.set_xlabel("Canal")
    ax.set_ylabel("Ventas ($)")
    ax.margins(y=0.12)
    ax.grid(axis="y", alpha=0.3)
    guardar_grafico("05_ventas_por_canal.png")


def grafico_utilidad_por_sucursal(tabla):
    tabla = tabla.sort_values("Utilidad", ascending=True)
    fig, ax = plt.subplots(figsize=(11, 7))
    barras = ax.barh(tabla["Sucursal"], tabla["Utilidad"], color="indianred")

    ax.bar_label(barras, labels=[formato_monto(v) for v in tabla["Utilidad"]], padding=3, fontsize=9)

    ax.set_title("Utilidad por Sucursal", fontsize=14, fontweight="bold")
    ax.set_xlabel("Utilidad ($)")
    ax.set_ylabel("Sucursal")
    ax.margins(x=0.15)
    ax.grid(axis="x", alpha=0.3)
    guardar_grafico("06_utilidad_por_sucursal.png")

In [97]:
# Reales vs meta (etiquetas rotadas por espacio, al ser barras agrupadas)
def grafico_ventas_reales_vs_meta(resumen_metas):
    tabla = resumen_metas.sort_values("SucursalID")
    x = range(len(tabla))
    ancho = 0.35

    fig, ax = plt.subplots(figsize=(12, 7))
    barras_meta = ax.bar([i - ancho/2 for i in x], tabla["Meta_Ventas_Total"], width=ancho, label="Meta", color="lightgray")
    barras_real = ax.bar([i + ancho/2 for i in x], tabla["Ventas_Reales_Total"], width=ancho, label="Real", color="royalblue")

    ax.bar_label(barras_meta, labels=[formato_monto(v) for v in tabla["Meta_Ventas_Total"]],
                 padding=3, fontsize=7, rotation=90)
    ax.bar_label(barras_real, labels=[formato_monto(v) for v in tabla["Ventas_Reales_Total"]],
                 padding=3, fontsize=7, rotation=90)

    ax.set_title("Ventas Reales vs Meta de Ventas por Sucursal", fontsize=14, fontweight="bold")
    ax.set_xlabel("Sucursal")
    ax.set_ylabel("Ventas ($)")
    ax.set_xticks(list(x))
    ax.set_xticklabels(tabla["SucursalID"], rotation=45, ha="right")
    ax.margins(y=0.2)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    guardar_grafico("07_ventas_reales_vs_meta.png")


def grafico_unidades_reales_vs_meta(resumen_metas):
    tabla = resumen_metas.sort_values("SucursalID")
    x = range(len(tabla))
    ancho = 0.35

    fig, ax = plt.subplots(figsize=(12, 7))
    barras_meta = ax.bar([i - ancho/2 for i in x], tabla["Meta_Unidades_Total"], width=ancho, label="Meta", color="lightgray")
    barras_real = ax.bar([i + ancho/2 for i in x], tabla["Unidades_Reales_Total"], width=ancho, label="Real", color="mediumseagreen")

    ax.bar_label(barras_meta, labels=[formato_unidades(v) for v in tabla["Meta_Unidades_Total"]],
                 padding=3, fontsize=7, rotation=90)
    ax.bar_label(barras_real, labels=[formato_unidades(v) for v in tabla["Unidades_Reales_Total"]],
                 padding=3, fontsize=7, rotation=90)

    ax.set_title("Unidades Reales vs Meta de Unidades por Sucursal", fontsize=14, fontweight="bold")
    ax.set_xlabel("Sucursal")
    ax.set_ylabel("Unidades")
    ax.set_xticks(list(x))
    ax.set_xticklabels(tabla["SucursalID"], rotation=45, ha="right")
    ax.margins(y=0.2)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    guardar_grafico("08_unidades_reales_vs_meta.png")

In [98]:
# Función principal (igual que antes, no cambia)
def main():
    crear_carpeta_graficos()
    base, ventas_por_categoria, top10_productos, top10_vendedores, ventas_por_canal, utilidad_por_sucursal, resumen_metas = cargar_datos()

    print("[2] Generando graficos...")
    grafico_ventas_por_mes_por_anio(base)
    grafico_ventas_por_categoria(ventas_por_categoria)
    grafico_top10_productos(top10_productos)
    grafico_top10_vendedores(top10_vendedores)
    grafico_ventas_por_canal(ventas_por_canal)
    grafico_utilidad_por_sucursal(utilidad_por_sucursal)
    grafico_ventas_reales_vs_meta(resumen_metas)
    grafico_unidades_reales_vs_meta(resumen_metas)

    archivos_generados = sorted(os.listdir(CARPETA_GRAFICOS))
    print(f"\n[3] Total de graficos generados: {len(archivos_generados)}")
    for archivo in archivos_generados:
        print(f"     - {archivo}")


if __name__ == "__main__":
    main()

[1] Carpeta de graficos lista en: /content/sample_data/graficos
[2] Generando graficos...
     -> Guardado: /content/sample_data/graficos/01_ventas_por_mes_por_anio.png
     -> Guardado: /content/sample_data/graficos/02_ventas_por_categoria.png
     -> Guardado: /content/sample_data/graficos/03_top10_productos.png
     -> Guardado: /content/sample_data/graficos/04_top10_vendedores.png
     -> Guardado: /content/sample_data/graficos/05_ventas_por_canal.png
     -> Guardado: /content/sample_data/graficos/06_utilidad_por_sucursal.png
     -> Guardado: /content/sample_data/graficos/07_ventas_reales_vs_meta.png
     -> Guardado: /content/sample_data/graficos/08_unidades_reales_vs_meta.png

[3] Total de graficos generados: 8
     - 01_ventas_por_mes_por_anio.png
     - 02_ventas_por_categoria.png
     - 03_top10_productos.png
     - 04_top10_vendedores.png
     - 05_ventas_por_canal.png
     - 06_utilidad_por_sucursal.png
     - 07_ventas_reales_vs_meta.png
     - 08_unidades_reales_vs_meta.

## Exportación automática de resultados

In [99]:
# Configuración y carga
import pandas as pd

ARCHIVO_BASE = "/content/sample_data/base_analitica_transformada.xlsx"
ARCHIVO_TABLAS = "/content/sample_data/tablas_resumen.xlsx"
ARCHIVO_METAS = "/content/sample_data/cumplimiento_metas.xlsx"
ARCHIVO_SALIDA = "/content/sample_data/reporte_final_analisis_comercial.xlsx"


def cargar_datos():
    base = pd.read_excel(ARCHIVO_BASE)
    tablas = pd.read_excel(ARCHIVO_TABLAS, sheet_name=None)  # dict con todas las hojas
    metas_detalle = pd.read_excel(ARCHIVO_METAS, sheet_name="Detalle_Mensual")
    metas_resumen = pd.read_excel(ARCHIVO_METAS, sheet_name="Resumen_por_Sucursal")
    return base, tablas, metas_detalle, metas_resumen

In [100]:
# Crear tabla resumen de KPIs
def crear_tabla_kpis(base):
    """Recalcula los KPIs principales directamente desde la base, para que
    el reporte final sea autocontenido y no dependa de una corrida anterior."""
    ventas_totales = base["Venta_Neta"].sum()
    utilidad_total = base["Utilidad"].sum()
    numero_ventas = base["VentaID"].nunique()

    kpis = {
        "Ventas Totales": ventas_totales,
        "Utilidad Total": utilidad_total,
        "Margen Promedio Simple (%)": base["Margen_%"].mean(),
        "Margen Promedio Ponderado (%)": utilidad_total / ventas_totales * 100,
        "Cantidad Vendida": base["Cantidad"].sum(),
        "Número de Ventas": numero_ventas,
        "Ticket Promedio": ventas_totales / numero_ventas,
        "Descuento Promedio (%)": base["Descuento"].mean() * 100,
    }

    tabla_kpis = pd.DataFrame(list(kpis.items()), columns=["Indicador", "Valor"])
    tabla_kpis["Valor"] = tabla_kpis["Valor"].round(2)
    print(f"[1] Tabla de KPIs creada ({len(tabla_kpis)} indicadores)")
    return tabla_kpis

In [101]:
# Exportar todo con ExcelWriter
def exportar_reporte(base, tabla_kpis, tablas, metas_detalle, metas_resumen):
    """
    Un unico archivo Excel con todas las piezas del proyecto, cada una en su
    propia hoja, usando ExcelWriter para escribir todo en un solo archivo.
    """
    # Mapeo entre el nombre de hoja pedido en la pauta y la hoja real en tablas_resumen.xlsx
    mapa_tablas = {
        "Ventas_Mes": "Ventas_por_Mes",
        "Ventas_Categoria": "Ventas_por_Categoria",
        "Ventas_Pais": "Ventas_por_Pais",
        "Ventas_Canal": "Ventas_por_Canal",
        "Top_Productos": "Top10_Productos",
        "Top_Vendedores": "Top10_Vendedores",
        "Utilidad_Sucursal": "Utilidad_por_Sucursal",
    }

    with pd.ExcelWriter(ARCHIVO_SALIDA) as writer:
        base.to_excel(writer, sheet_name="Base_Analitica_Final", index=False)
        print("[2] Base analítica final exportada")

        tabla_kpis.to_excel(writer, sheet_name="KPIs", index=False)
        print("[3] Tabla de KPIs exportada")

        for nombre_hoja_final, nombre_hoja_origen in mapa_tablas.items():
            tablas[nombre_hoja_origen].to_excel(writer, sheet_name=nombre_hoja_final, index=False)
        print(f"[4] Tablas resumen exportadas: {list(mapa_tablas.keys())}")

        metas_resumen.to_excel(writer, sheet_name="Cumplimiento_Metas", index=False)
        metas_detalle.to_excel(writer, sheet_name="Cumplimiento_Metas_Detalle", index=False)
        print("[5] Tabla de cumplimiento de metas exportada (resumen + detalle mensual)")

    print(f"\n[6] Archivo final guardado en: {ARCHIVO_SALIDA}")

In [102]:
# Validación final con print()
def validar_exportacion():
    """Reabre el archivo generado y verifica que todas las hojas esperadas existan
    y tengan contenido, usando print() para dejar constancia visible."""
    hojas_esperadas = [
        "Base_Analitica_Final", "KPIs", "Ventas_Mes", "Ventas_Categoria",
        "Ventas_Pais", "Ventas_Canal", "Top_Productos", "Top_Vendedores",
        "Utilidad_Sucursal", "Cumplimiento_Metas", "Cumplimiento_Metas_Detalle",
    ]

    archivo = pd.ExcelFile(ARCHIVO_SALIDA)
    print("\n" + "=" * 55)
    print("VALIDACION FINAL DE LA EXPORTACION")
    print("=" * 55)

    for hoja in hojas_esperadas:
        assert hoja in archivo.sheet_names, f"Falta la hoja '{hoja}' en el archivo final"
        df_hoja = pd.read_excel(ARCHIVO_SALIDA, sheet_name=hoja)
        print(f"  ✓ {hoja:<28} {df_hoja.shape[0]:>5} filas x {df_hoja.shape[1]} columnas")

    print("=" * 55)
    print("Todas las hojas esperadas estan presentes y con datos. OK.")

In [103]:
# Función principal
def main():
    base, tablas, metas_detalle, metas_resumen = cargar_datos()
    tabla_kpis = crear_tabla_kpis(base)
    exportar_reporte(base, tabla_kpis, tablas, metas_detalle, metas_resumen)
    validar_exportacion()


if __name__ == "__main__":
    main()

[1] Tabla de KPIs creada (8 indicadores)
[2] Base analítica final exportada
[3] Tabla de KPIs exportada
[4] Tablas resumen exportadas: ['Ventas_Mes', 'Ventas_Categoria', 'Ventas_Pais', 'Ventas_Canal', 'Top_Productos', 'Top_Vendedores', 'Utilidad_Sucursal']
[5] Tabla de cumplimiento de metas exportada (resumen + detalle mensual)

[6] Archivo final guardado en: /content/sample_data/reporte_final_analisis_comercial.xlsx

VALIDACION FINAL DE LA EXPORTACION
  ✓ Base_Analitica_Final          1800 filas x 43 columnas
  ✓ KPIs                             8 filas x 2 columnas
  ✓ Ventas_Mes                      15 filas x 6 columnas
  ✓ Ventas_Categoria                 6 filas x 5 columnas
  ✓ Ventas_Pais                      5 filas x 4 columnas
  ✓ Ventas_Canal                     5 filas x 4 columnas
  ✓ Top_Productos                   10 filas x 5 columnas
  ✓ Top_Vendedores                  10 filas x 5 columnas
  ✓ Utilidad_Sucursal               12 filas x 6 columnas
  ✓ Cumplimiento_Met